### LoRa FineTuning

***What is LoRa?***

LoRA adds small, trainable matrices (the "overlays") alongside the large, frozen original model weights. During fine-tuning, only these small LoRA matrices are updated, not the full model. This makes training much faster, uses less memory, and results in smaller saved models.

In [ ]:
from datasets import load_dataset, concatenate_datasets, DatasetDict
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, clear_output
from scipy.signal import resample
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import WhisperTokenizer, WhisperFeatureExtractor, WhisperForConditionalGeneration
import evaluate
from jiwer import cer
import pandas as pd
from datasets import Dataset, Audio
import os

In [ ]:
from peft import LoraConfig, get_peft_model
from peft import PeftModel

#### Resample the audios

In [2]:
def down_sample_audio(audio_original, original_sample_rate):
    target_sample_rate = 16000
    num_samples = int(len(audio_original) * target_sample_rate / original_sample_rate)
    downsampled_audio = resample(audio_original, num_samples)
    return downsampled_audio

#### Load Base Model & Processor

In [3]:
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language='ar', task='transcribe')
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small", language='ar', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to('cuda')

c:\Users\cyrine.anene_amaris\.virtualenvs\AI-Enhanced-Eligibility-Checker-Mo6Xk0zX\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


#### LoRA Adaptation

In [ ]:
model.gradient_checkpointing_enable()
model.config.use_cache = False
lora_config = LoraConfig(
    r=32, #LoRA attention dimension. # Higher 'r' means more trainable parameters, potentially better performance, but more memory.
    lora_alpha=64, #LoRA scaling factor. Typically set to 2 * r.
    target_modules=["q_proj", "v_proj"], #Which layers to apply LoRA to. # For Whisper, 'q_proj' (query) and 'v_proj' (value) are common.
    lora_dropout=0.05, #Dropout probability for the LoRA layers.
    bias="none", #Whether to train bias terms. 'none' is common for LoRA.
    task_type="SEQ_2_SEQ_LM", #This should be "SEQ_2_SEQ_LM" for sequence-to-sequence language modeling like Whisper.
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

#### Load Data and Data Analysis

In [ ]:
CSV_PATH = "linto_segmented_dataset.csv"
df = pd.read_csv(CSV_PATH)
small_df = df.head(100)
small_df['audio_path'] = small_df['wav']

data_list_for_hf_dataset = []

for index, row in small_df.iterrows():
    data_list_for_hf_dataset.append({
        "audio": {"path": row['audio_path']}, 
        "sentence": row['wrd'] 
    })

custom_hf_dataset = Dataset.from_list(data_list_for_hf_dataset)

custom_hf_dataset = custom_hf_dataset.cast_column("audio", Audio())

split_custom_dataset = custom_hf_dataset.train_test_split(test_size=0.2, seed=42)

train_data = split_custom_dataset['train']
test_data = split_custom_dataset['test']

print(f"Loaded {len(train_data)} training examples and {len(test_data)} test examples from your local dataset for testing.")

In [ ]:
list_of_transcription_lengths = []
tokenized_text = tokenizer(train_data['sentence']).input_ids
for text in tokenized_text:
    list_of_transcription_lengths.append(len(text))

plt.hist(list_of_transcription_lengths)
plt.xlabel("sentence length")
plt.ylabel("number of transcripts")
plt.title("Distribution of Tokenized Transcription Lengths") 
plt.show()

#### Custom PyTorch Dataset

In [ ]:
MAX_SEQ_LEN = 70

class whisper_training_dataset(torch.utils.data.Dataset):
    def __init__(self, dataset, max_len):
        self.dataset = dataset
        self.max_len = max_len
        self.bos_token = model.config.decoder_start_token_id 

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        if 'audio' not in item or 'array' not in item['audio'] or 'sampling_rate' not in item['audio']:
           
            print(f"Skipping item {idx} due to missing audio data.")
            return None

        audio_data = down_sample_audio(item['audio']["array"], item['audio']["sampling_rate"])

        input_features = feature_extractor(raw_speech=audio_data, sampling_rate=16000, return_tensors='pt').input_features[0]

        transcription = item["sentence"]

        labels = tokenizer(transcription, padding="max_length", max_length=self.max_len, truncation=True, return_tensors="pt")
        labels = labels["input_ids"].masked_fill(labels['attention_mask'].ne(1), -100)
        labels = labels[0][1:] 

        return {
            "input_features": input_features,
            "labels": labels
        }

#### Data Loaders

In [ ]:
def collate_fn(batch):
    batch = [item for item in batch if item is not None] 
    if not batch:
        return {} 

    input_features = torch.stack([x['input_features'] for x in batch])
    labels = torch.stack([x['labels'] for x in batch])
    return {"input_features": input_features, "labels": labels}

train_whisper_dataset = whisper_training_dataset(dataset=train_data, max_len=70) 
train_dataloader = torch.utils.data.DataLoader(
    train_whisper_dataset,
    batch_size=8,  
    shuffle=True,  
    collate_fn=collate_fn 
)

#### Evaluation Function

In [ ]:
def evaluation(model_eval): 
    device='cuda'

    test_dataset = whisper_training_dataset(dataset=test_data, max_len=MAX_SEQ_LEN) 
    test_dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=8,
        shuffle=False, 
        collate_fn=collate_fn
    )

    model_eval.eval() 
    predictions=[]
    references=[]

    print("Running evaluation...")
    for batch_idx, batch in enumerate(tqdm(test_dataloader, total=len(test_dataloader), desc="Evaluating")):
        if not batch: 
            continue

        input_features, labels = batch["input_features"].to(device), batch["labels"].to(device)

        with torch.no_grad():
          
            generated_tokens = model_eval.generate(
                input_features=input_features,
                language='arabic',
                task='transcribe',
                max_new_tokens=MAX_SEQ_LEN                               
            )

        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

        labels = labels.cpu().numpy()
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id) 
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        predictions.extend(decoded_preds)
        references.extend(decoded_labels)

    final_wer = cer(predictions=predictions, references=references) * 100 
    print(f"\nEvaluation Complete. Final CER: {final_wer:.2f}%")
    return final_wer

#### Training Loop

In [ ]:
torch.cuda.empty_cache()
model.train() 
device='cuda'
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5) 

max_epochs = 20 
running_loss = []
running_wer_per_epoch = [] 

# === Early Stopping Parameters ===
best_cer = float('inf')
patience_counter = 0
patience = 5 

print("Starting LoRA fine-tuning...")
for epoch in range(max_epochs):
    print(f"\nEpoch {epoch + 1}/{max_epochs}")
    model.train() 
    for batch_idx, batch in enumerate(tqdm(train_dataloader, total=len(train_dataloader), leave=False, desc=f"Epoch {epoch + 1}")):
        if not batch:
            continue

        input_features, labels = batch["input_features"].to(device), batch["labels"].to(device)

        outputs = model(input_features, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad() 

        running_loss.append(loss.item())

        if (batch_idx + 1) % 50 == 0:
            clear_output(wait=True)
            plt.figure(figsize=(10, 4))
            plt.plot(running_loss)
            plt.xlabel('Steps')
            plt.ylabel('Loss')
            plt.title("Training Loss Over Steps")
            plt.grid(True)
            plt.show()
            print(f"Epoch {epoch + 1}, Step {batch_idx + 1}/{len(train_dataloader)}, Loss: {loss.item():.4f}")

        if (batch_idx + 1) % 100 == 0:
            save_path = f'lora_checkpoints/lora_model_epoch_{epoch+1}_step_{batch_idx+1}'
            os.makedirs(os.path.dirname(save_path), exist_ok=True) # Ensure directory exists
            model.save_pretrained(save_path)
            print(f"LoRA adapters saved to '{save_path}'")

    torch.cuda.empty_cache()

    # --- Evaluate after each epoch ---
    current_cer = evaluation(model)
    running_wer_per_epoch.append(current_cer)
    print(f"End of Epoch {epoch + 1}, Validation CER: {current_cer:.2f}%")

    # --- Early Stopping Logic ---
    if current_cer < best_cer:
        best_cer = current_cer
        patience_counter = 0 

        best_model_save_path = 'best_lora_model'
        model.save_pretrained(best_model_save_path)
        print(f"New best model saved to '{best_model_save_path}' with CER: {best_cer:.2f}%")
    else:
        patience_counter += 1
        print(f"Validation CER did not improve. Patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"Early stopping triggered! No improvement for {patience} consecutive epochs.")
        break 

print("\nLoRA Fine-tuning complete!")

# --- Final Plots ---
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(running_wer_per_epoch) + 1), running_wer_per_epoch, marker='o', linestyle='-')
plt.xlabel('Epoch')
plt.ylabel('Validation CER (%)')
plt.title('Validation CER Over Epochs')
plt.grid(True)
plt.show()

print("\nCER values per epoch:", running_wer_per_epoch)

#### Post-Training Evaluation

In [ ]:
print("\n--- Loading best model for final evaluation and inference ---")
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to('cuda')
model_for_final_eval = PeftModel.from_pretrained(base_model, "best_lora_model")

model_for_final_eval.eval()
print("\n--- Final Evaluation ---")
final_cer_best_model = evaluation(model_for_final_eval)
print(f"Final Character Error Rate (from best saved model): {final_cer_best_model:.2f}%")

#### Inference Examples

In [ ]:
print("\n--- Inference Examples ---")
model.eval()
for idx in range(5):
    target = test_data[idx]['sentence']
    audio_original = test_data[idx]['audio']['array']
    original_sample_rate = test_data[idx]['audio']['sampling_rate']

    audio_16000 = down_sample_audio(audio_original, original_sample_rate)
    input_feature = feature_extractor(raw_speech=audio_16000,
                                      sampling_rate=16000,
                                      return_tensors='pt').input_features

    with torch.no_grad():
        op = model.generate(input_feature.to('cuda'),
                            language='arabic', 
                            task='transcribe',
                            max_new_tokens=MAX_SEQ_LEN) 

    text_pred =  tokenizer.batch_decode(op, skip_special_tokens=True)[0]

    print(f'------- Example {idx+1} ------')
    print(f'True : {target} \nPred : {text_pred}')
    print('\n')